# LSG50 Index — Pipeline Construction

Este notebook construye y persiste el índice **LSG50**, un índice propietario basado en métricas fundamentales.

El pipeline sigue los siguientes pasos:

- Construcción del universo del S&P 500
- Cálculo de métricas fundamentales
- Construcción del índice LSG50
- Asignación de pesos
- Persistencia de resultados en la base de datos SQLite

In [ ]:
#Import and Config

# Core
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from datetime import date
import sys
sys.path.append("../src")

from fundamentals import fetch_fundamentals

# Config
DB_PATH = "sqlite:///lsg50.db"
TOP_N = 50

engine = create_engine(DB_PATH)
today = str(date.today())

print("Pipeline date:", today)

## 1. Database Schema Initialization

En esta sección se inicializan las tablas necesarias para almacenar la información del pipeline.

Se crean tres tablas principales:

- **universe**: contiene las empresas del S&P 500 y su sector
- **fundamentals_snapshot**: almacena las métricas fundamentales calculadas en cada ejecución
- **lsg50_composition**: guarda la composición del índice en cada fecha de cálculo

Esto permite mantener un histórico del índice y sus métricas.

In [ ]:
with engine.connect() as conn:

    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS universe (
        symbol TEXT PRIMARY KEY,
        name TEXT,
        sector TEXT
    );
    """))

    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS fundamentals_snapshot (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT,
        calculation_date TEXT,
        market_cap REAL,
        net_income_cagr_5y REAL,
        growth_pct REAL,
        size_pct REAL,
        cfo_index REAL,
        FOREIGN KEY(symbol) REFERENCES universe(symbol)
    );
    """))

    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS lsg50_composition (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT,
        calculation_date TEXT,
        weight REAL,
        cfo_index REAL,
        FOREIGN KEY(symbol) REFERENCES universe(symbol)
    );
    """))

print("Schema ready.")

## 2. Build S&P 500 Universe

Aquí se construye el universo de empresas utilizando la lista pública del **S&P 500** disponible en Wikipedia.

Se extraen:

- símbolo
- nombre de la compañía
- sector (GICS)

Posteriormente, el universo se almacena en la tabla **universe** de la base de datos.

In [ ]:
def build_universe():

    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    df_sp500 = pd.read_html(response.text)[0]

    df_sp500["symbol"] = (
        df_sp500["Symbol"]
        .str.replace(".", "-", regex=False)
        .str.strip()
    )

    df_universe = df_sp500[["symbol", "Security", "GICS Sector"]].rename(
        columns={
            "Security": "name",
            "GICS Sector": "sector"
        }
    )

    return df_universe

df_universe = build_universe()
df_universe.to_sql("universe", engine, if_exists="replace", index=False)

print("Universe stored:", len(df_universe), "companies")

## 3. Fundamental Metrics Calculation

En esta sección se calculan las métricas fundamentales necesarias para construir el índice.

Para cada empresa del universo se obtienen:

- Market Capitalization
- Net Income CAGR (5 años)

Posteriormente se calculan percentiles relativos dentro del universo:

- **growth_pct** → percentil de crecimiento
- **size_pct** → percentil de tamaño

Estas métricas se combinan para construir el **CFO Index**, que será utilizado para rankear las empresas.

In [ ]:
def compute_percentiles(df):

    df["growth_pct"] = df["net_income_cagr_5y"].rank(pct=True)
    df["size_pct"] = df["market_cap"].rank(pct=True)

    df["cfo_index"] = 0.5 * df["growth_pct"] + 0.5 * df["size_pct"]

    return df

symbols = df_universe["symbol"].tolist()

results = []

for s in symbols:
    data = fetch_fundamentals(s)
    if data is not None:
        results.append(data)

df_master = pd.DataFrame(results)

df_master = compute_percentiles(df_master)

df_master["calculation_date"] = today

cols = [
    "symbol",
    "calculation_date",
    "market_cap",
    "net_income_cagr_5y",
    "growth_pct",
    "size_pct",
    "cfo_index"
]

df_master[cols].to_sql(
    "fundamentals_snapshot",
    engine,
    if_exists="append",
    index=False
)

print("Fundamental snapshot stored:", len(df_master))

## 4. LSG50 Index Construction

En esta etapa se construye el índice LSG50.

Metodología:

1. Rankear empresas según el **CFO Index**
2. Seleccionar las **50 empresas con mayor puntuación**
3. Construir el índice a partir de estas compañías

Este proceso genera la composición base del índice.

In [ ]:
TOP_N = 50

df_index = (
    df_master
    .sort_values("cfo_index", ascending=False)
    .head(TOP_N)
    .copy()
)

print("Selected constituents:", len(df_index))
df_index[["symbol", "cfo_index"]].head()

### 4.2 Weight Allocation

En lugar de utilizar ponderación igualitaria o por capitalización bursátil (como el S&P 500),
el índice LSG50 asigna pesos proporcionalmente al **CFO Index** de cada empresa.

Esto refuerza la lógica de selección basada en fundamentales y genera una construcción diferenciada del índice.

In [ ]:
df_index["weight"] = df_index["cfo_index"] / df_index["cfo_index"].sum()

print("Weight sum check:", df_index["weight"].sum())
df_index[["symbol", "cfo_index", "weight"]].head()

### 4.3 Sector Exposure (Sanity Check)

Antes de persistir el índice, analizamos la distribución sectorial de las empresas seleccionadas.

Este paso permite verificar que el índice mantiene una exposición diversificada
y no presenta concentraciones excesivas en un solo sector.

In [ ]:
df_index = df_index.merge(
    df_universe[["symbol", "sector"]],
    on="symbol",
    how="left"
)

df_index["sector"].value_counts(normalize=True)

### 4.4 Persist Index Composition

Finalmente, la composición del índice se guarda en la tabla **lsg50_composition**.

Cada registro incluye:

- símbolo
- fecha de cálculo
- peso asignado
- puntuación del CFO Index

Esto permite mantener un historial de composiciones del índice a lo largo del tiempo.

In [ ]:
df_index["calculation_date"] = today

df_index[["symbol", "calculation_date", "weight", "cfo_index"]].to_sql(
    "lsg50_composition",
    engine,
    if_exists="append",
    index=False
)

print("LSG50 composition stored:", len(df_index))

In [ ]:
df_index["weight"].sum()

---

## LSG50 Construction Complete

The index has been successfully constructed and persisted.

Key characteristics:
- Top 50 by proprietary CFO Index
- Score-weighted allocation
- Snapshot stored for historical tracking

In [ ]:
#Verificación de tablas existentes
import pandas as pd

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", engine)

In [ ]:
#Últimos fundamentals guardados
pd.read_sql("""
SELECT *
FROM fundamentals_snapshot
ORDER BY calculation_date DESC
LIMIT 10
""", engine)

In [ ]:
#Composición del índice 
pd.read_sql("""
SELECT *
FROM lsg50_composition
ORDER BY weight DESC
LIMIT 10
""", engine)

In [ ]:
pd.read_sql("""
SELECT COUNT(*) as n_constituents
FROM lsg50_composition
""", engine)

In [ ]:
df_current = pd.read_sql("""
SELECT *
FROM lsg50_composition
WHERE calculation_date = (
    SELECT MAX(calculation_date)
    FROM lsg50_composition
)
""", engine)

df_current.head()

In [ ]:
df_current.sort_values("weight", ascending=False).head(10)

In [ ]:
top10_weight = (
    df_current
    .sort_values("weight", ascending=False)
    .head(10)["weight"]
    .sum()
)

top10_weight

Este índice se desconcentra en relación al S&P500, donde las 10 empresas más grandes suponen el 30-35% del mismo vs un 22% del propio.

In [ ]:
#Concentración real del índice para medir la diversificación
import numpy as np

herfindahl = np.sum(df_current["weight"] ** 2)

herfindahl

In [ ]:
df_current = pd.read_sql("""
SELECT *
FROM lsg50_composition
WHERE calculation_date = (
    SELECT MAX(calculation_date)
    FROM lsg50_composition
)
""", engine)

df_current = df_current.drop_duplicates(subset=["symbol"])

In [ ]:
len(df_current)

Se observa un predominio de empresas vinculadas con TICS, financieras, industriales y de salud.

In [ ]:
#Creamos dataset de donde obtendremos el índice

df_index_final = (
    df_current
    .merge(df_universe[["symbol", "sector"]], on="symbol", how="left")
    [["symbol", "sector", "cfo_index", "weight", "calculation_date"]]
)

df_index_final.head()

In [ ]:
df_index_final.to_sql(
    "lsg50_index_members",
    engine,
    if_exists="append",
    index=False
)

print("LSG50 index members stored:", len(df_index_final))

In [ ]:
pd.read_sql("""
SELECT *
FROM lsg50_index_members
ORDER BY weight DESC
LIMIT 10
""", engine)

In [ ]:
pd.read_sql("""
SELECT 
calculation_date,
COUNT(*) as n_constituents,
SUM(weight) as total_weight
FROM lsg50_index_members
GROUP BY calculation_date
ORDER BY calculation_date DESC
""", engine)

## 5. LSG50 Portfolio Analysis

En esta sección analizamos la estructura del índice construido.

Se estudian tres aspectos principales:

- principales componentes del índice
- exposición sectorial
- distribución de pesos

Este análisis permite evaluar la diversificación del índice y su estructura interna.

### 5.1 Top Holdings

Primero analizamos las empresas con mayor peso dentro del índice.

Este análisis permite identificar los principales contribuyentes al rendimiento potencial del LSG50.

In [ ]:
top10 = (
    df_current
    .sort_values("weight", ascending=False)
    .head(10)
)

top10[["symbol", "weight", "cfo_index"]]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.bar(
    top10["symbol"],
    top10["weight"]
)

plt.title("LSG50 - Top 10 Holdings by Weight")
plt.xlabel("Company")
plt.ylabel("Weight")

plt.show()

### 5.2 Sector Allocation

En esta sección analizamos la distribución sectorial del índice.

El objetivo es entender las exposiciones estructurales del LSG50 y compararlas
con el universo más amplio del S&P 500.

In [ ]:
import sqlite3

conn = sqlite3.connect("lsg50.db")

In [ ]:
composition = pd.read_sql("""
SELECT symbol, weight
FROM lsg50_composition
""", conn)

In [ ]:
composition = pd.read_sql("""
SELECT symbol, weight
FROM lsg50_composition
WHERE calculation_date = (
    SELECT MAX(calculation_date)
    FROM lsg50_composition
)
""", conn)

In [ ]:
composition = pd.read_sql("""
SELECT
    c.symbol,
    u.sector,
    c.weight,
    c.cfo_index,
    c.calculation_date
FROM lsg50_composition c
JOIN universe u
ON c.symbol = u.symbol
WHERE c.calculation_date = (
    SELECT MAX(calculation_date)
    FROM lsg50_composition
)
""", conn)

composition = composition.drop_duplicates(subset="symbol")

composition.head()

In [ ]:
# Top holdings
top10 = composition.sort_values("weight", ascending=False).head(10)

# Sector allocation
sector_weights = composition.groupby("sector")["weight"].sum().sort_values(ascending=False)

print("Top 10 holdings:")
display(top10[["symbol", "weight", "cfo_index"]])

print("\nSector allocation:")
display(sector_weights)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

sector_weights.sort_values().plot(kind="barh")

plt.title("LSG50 — Sector Allocation")
plt.xlabel("Weight")
plt.ylabel("Sector")

plt.show()

In [ ]:
composition["weight"].sum()

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(top10["symbol"], top10["weight"])

plt.title("LSG50 — Top 10 Holdings by Weight")
plt.xlabel("Company")
plt.ylabel("Weight")

plt.show()

### 5.3 Weight Distribution

Aquí analizamos cómo se distribuyen los pesos entre los componentes del índice.

Una distribución más equilibrada suele indicar mayor diversificación
y menor riesgo de concentración.

In [ ]:
df_current["weight"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(composition["weight"], bins=20)

plt.title("LSG50 — Weight Distribution")
plt.xlabel("Weight")
plt.ylabel("Number of Companies")

plt.show()

## LSG50 Construction Complete

El índice LSG50 ha sido construido correctamente y almacenado en la base de datos.

Características principales:

- Selección de las **50 empresas con mayor CFO Index**
- Ponderación basada en métricas fundamentales
- Persistencia de resultados para análisis histórico